# Chapter 23 — Recurrent Neural Networks I: RNNs, LSTM, GRU & BPTT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part3_neural_networks/ch23_recurrent_neural_networks_i/ch23_notebook.ipynb)

Companion notebook for **Chapter 23** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- RNNs process a sequence one element at a time, carrying a hidden state that acts as memory of everything seen so far.
- The same weight matrices U, W and V are reused at every timestep, so model size does not grow with sequence length.
- Unfolding the network across time turns training into ordinary backpropagation, known as BPTT.
- Plain RNNs cannot learn long-range dependencies because gradients shrink with every step back through time.
- The LSTM's additive cell state, controlled by forget, input and output gates, preserves those gradients; the GRU is a lighter alternative.

## Example 1: Vanilla RNN Forward Pass from Scratch

This builds a vanilla RNN with the three weight matrices U, W, V and runs a length-4 sequence through it. You can watch the hidden state h_t evolve at each timestep - it depends on the current input and the previous hidden state, which is the network's memory. The same weights are reused at every timestep (weight sharing in time).

In [1]:
# Chapter 23 - Example 1: Vanilla RNN forward pass from scratch
import numpy as np
np.random.seed(1)

def tanh(x): return np.tanh(x)

print("=== Vanilla RNN Forward Pass (from scratch) ===\n")
print("Architecture: input -> hidden (recurrent) -> output")
print("  h_t = tanh(U x_t + W h_(t-1) + b_h)")
print("  y_t = V h_t + b_o\n")

# Dimensions
d = 3   # input size
J = 4   # hidden size
K = 2   # output size

# Weight matrices (the 3 sets that define an RNN)
U = np.random.randn(J, d) * 0.5   # input -> hidden
W = np.random.randn(J, J) * 0.5   # hidden -> hidden (recurrent)
V = np.random.randn(K, J) * 0.5   # hidden -> output
b_h = np.zeros(J)
b_o = np.zeros(K)

print(f"U (input->hidden) : {U.shape}")
print(f"W (recurrent)     : {W.shape}")
print(f"V (hidden->output): {V.shape}\n")

# An input sequence of length T=4 (e.g., 4 words, each a 3-dim vector)
sequence = [np.array([1.0, 0.5, -0.5]),
            np.array([0.2, -1.0, 0.3]),
            np.array([-0.5, 0.8, 1.0]),
            np.array([0.9, 0.1, -0.2])]

print("Processing a sequence of length T=4:\n")
h = np.zeros(J)   # initial hidden state h_0
print(f"Initial hidden state h_0: {np.round(h, 3)}\n")

for t, x in enumerate(sequence, 1):
    h = tanh(U @ x + W @ h + b_h)
    y = V @ h + b_o
    print(f"t={t}: x_t={np.round(x,2)}")
    print(f"     h_{t} = {np.round(h, 4)}")
    print(f"     y_{t} = {np.round(y, 4)}")
    print()

print("Key insight: h_t depends on x_t AND h_(t-1), so the hidden state")
print("carries 'memory' of ALL previous inputs in the sequence.")
print("The SAME weights U, W, V are reused at every timestep (weight sharing in time).")

=== Vanilla RNN Forward Pass (from scratch) ===

Architecture: input -> hidden (recurrent) -> output
  h_t = tanh(U x_t + W h_(t-1) + b_h)
  y_t = V h_t + b_o

U (input->hidden) : (4, 3)
W (recurrent)     : (4, 4)
V (hidden->output): (2, 4)

Processing a sequence of length T=4:

Initial hidden state h_0: [0. 0. 0. 0.]

t=1: x_t=[ 1.   0.5 -0.5]
     h_1 = [0.6591 0.2498 0.5387 0.6386]
     y_1 = [-0.335  -0.5169]

t=2: x_t=[ 0.2 -1.   0.3]
     h_2 = [ 0.1868 -0.6933  0.6564 -0.8292]
     y_2 = [-0.2714  0.0137]

t=3: x_t=[-0.5  0.8  1. ]
     h_3 = [ 0.0163 -0.443  -0.7587  0.2783]
     y_3 = [0.0875 0.4345]

t=4: x_t=[ 0.9  0.1 -0.2]
     h_4 = [0.2472 0.0487 0.1787 0.2374]
     y_4 = [-0.1291 -0.167 ]

Key insight: h_t depends on x_t AND h_(t-1), so the hidden state
carries 'memory' of ALL previous inputs in the sequence.
The SAME weights U, W, V are reused at every timestep (weight sharing in time).


## Example 2: LSTM Cell from Scratch

This computes one full timestep of an LSTM cell: the forget, input, and output gates (sigmoid), the candidate state (tanh), the additive cell-state update, and the hidden-state read-out. The additive cell-state update is what lets gradients survive over long sequences. The parameter count confirms an LSTM has ~4x the weights of a vanilla RNN layer.

In [2]:
# Chapter 23 - Example 2: LSTM cell from scratch (one timestep, all gates)
import numpy as np
np.random.seed(2)

def sigmoid(x): return 1/(1+np.exp(-x))

print("=== LSTM Cell - One Timestep, All Gates ===\n")
print("An LSTM has 3 gates + a cell state, designed to avoid vanishing gradients:")
print("  Forget gate f_t : what to erase from cell state")
print("  Input gate  i_t : what new info to write")
print("  Output gate o_t : what to read out as the hidden state")
print("  Cell state  s_t : the long-term memory conveyor belt\n")

d = 3   # input size
J = 4   # hidden/cell size

# Each gate has its own U (input) and W (recurrent) weight matrices
def rand(*shape): return np.random.randn(*shape) * 0.3
U_i, W_i, b_i = rand(J,d), rand(J,J), np.zeros(J)   # input gate
U_f, W_f, b_f = rand(J,d), rand(J,J), np.zeros(J)   # forget gate
U_o, W_o, b_o = rand(J,d), rand(J,J), np.zeros(J)   # output gate
U_c, W_c, b_c = rand(J,d), rand(J,J), np.zeros(J)   # candidate (temporary state)

# Previous states
x_t   = np.array([0.5, -0.3, 0.8])   # current input
h_prev = np.array([0.1, -0.2, 0.05, 0.15])   # previous hidden state
s_prev = np.array([0.2, 0.1, -0.1, 0.3])     # previous cell state

print(f"Current input  x_t   : {x_t}")
print(f"Previous hidden h_(t-1): {h_prev}")
print(f"Previous cell   s_(t-1): {s_prev}\n")

# Gate computations
i_t = sigmoid(U_i @ x_t + W_i @ h_prev + b_i)
f_t = sigmoid(U_f @ x_t + W_f @ h_prev + b_f)
o_t = sigmoid(U_o @ x_t + W_o @ h_prev + b_o)
s_tilde = np.tanh(U_c @ x_t + W_c @ h_prev + b_c)   # candidate

print("Step 1 - Gate activations (sigmoid -> values in 0..1):")
print(f"  Input gate  i_t = {np.round(i_t, 3)}")
print(f"  Forget gate f_t = {np.round(f_t, 3)}")
print(f"  Output gate o_t = {np.round(o_t, 3)}")
print(f"  Candidate ~s_t  = {np.round(s_tilde, 3)}  (tanh, -1..1)\n")

# Cell state update: selective forget + selective write
s_t = f_t * s_prev + i_t * s_tilde
print("Step 2 - Cell state update (selective forget + selective write):")
print(f"  s_t = f_t (*) s_(t-1) + i_t (*) ~s_t")
print(f"      = {np.round(f_t*s_prev,3)} + {np.round(i_t*s_tilde,3)}")
print(f"      = {np.round(s_t, 4)}\n")

# Hidden state: selective read
h_t = o_t * np.tanh(s_t)
print("Step 3 - Hidden state (selective read):")
print(f"  h_t = o_t (*) tanh(s_t) = {np.round(h_t, 4)}\n")

print("Why no vanishing gradient? The cell state s_t updates ADDITIVELY")
print("(f_t * s_(t-1) + ...), so gradients flow back through time without")
print("being repeatedly multiplied down to zero - unlike a vanilla RNN.")

# Param count comparison
print(f"\nParameter count: LSTM has 4 weight sets (i,f,o,candidate)")
print(f"  = 4x the parameters of a vanilla RNN hidden layer")
print(f"  Vanilla RNN: U({J}x{d}) + W({J}x{J}) = {J*d + J*J} weights")
print(f"  LSTM       : 4 x ({J*d + J*J}) = {4*(J*d + J*J)} weights")

=== LSTM Cell - One Timestep, All Gates ===

An LSTM has 3 gates + a cell state, designed to avoid vanishing gradients:
  Forget gate f_t : what to erase from cell state
  Input gate  i_t : what new info to write
  Output gate o_t : what to read out as the hidden state
  Cell state  s_t : the long-term memory conveyor belt

Current input  x_t   : [ 0.5 -0.3  0.8]
Previous hidden h_(t-1): [ 0.1  -0.2   0.05  0.15]
Previous cell   s_(t-1): [ 0.2  0.1 -0.1  0.3]

Step 1 - Gate activations (sigmoid -> values in 0..1):
  Input gate  i_t = [0.373 0.531 0.469 0.576]
  Forget gate f_t = [0.433 0.658 0.505 0.515]
  Output gate o_t = [0.478 0.442 0.336 0.448]
  Candidate ~s_t  = [ 0.317  0.312 -0.061  0.537]  (tanh, -1..1)

Step 2 - Cell state update (selective forget + selective write):
  s_t = f_t (*) s_(t-1) + i_t (*) ~s_t
      = [ 0.087  0.066 -0.051  0.155] + [ 0.118  0.166 -0.029  0.309]
      = [ 0.2049  0.2314 -0.0794  0.464 ]

Step 3 - Hidden state (selective read):
  h_t = o_t (*) tan

## Example 3: RNN vs LSTM Gradient Flow

This traces the gradient backward through 30 timesteps. In a vanilla sigmoid RNN each step multiplies the gradient by a factor below 1, so it vanishes to ~1e-21. In an LSTM the cell-state highway keeps the gradient near its original size. This is the concrete reason LSTMs learn long-range dependencies that plain RNNs cannot.

In [3]:
# Chapter 23 - Example 3: RNN vs LSTM gradient flow over a long sequence
import numpy as np
np.random.seed(3)

def sigmoid(x): return 1/(1+np.exp(-x))
def sigmoid_deriv(x): return sigmoid(x)*(1-sigmoid(x))

print("=== Why Vanilla RNNs Forget: Gradient Flow Over Time ===\n")
print("During BPTT (Backprop Through Time), the gradient at the first timestep")
print("depends on a product of factors across ALL timesteps.\n")

T = 30  # sequence length

# Vanilla RNN: gradient multiplies by (W * sigmoid') each step back
# Typical sigmoid derivative <= 0.25, recurrent weight ~ moderate
W_rnn = 0.8
print(f"Vanilla RNN (recurrent weight W={W_rnn}, sigmoid neurons):")
grad = 1.0
trace_rnn = []
for t in range(T):
    deriv = 0.25  # max sigmoid derivative
    grad *= W_rnn * deriv
    trace_rnn.append(grad)
print(f"  Gradient reaching t=1 after {T} steps: {grad:.3e}")
print(f"  (each step multiplies by W*sigmoid' = {W_rnn*0.25} < 1, so it vanishes)\n")

# LSTM: cell-state path has near-1 multiplier (forget gate ~ 1)
print("LSTM (cell-state highway, forget gate f_t ~ 0.95):")
grad_lstm = 1.0
f_gate = 0.95
trace_lstm = []
for t in range(T):
    grad_lstm *= f_gate   # gradient flows through additive cell state
    trace_lstm.append(grad_lstm)
print(f"  Gradient reaching t=1 after {T} steps: {grad_lstm:.3e}")
print(f"  (cell state path multiplies by ~f_t={f_gate}, close to 1, so it survives)\n")

print(f"{'Timestep back':>14} {'RNN gradient':>16} {'LSTM gradient':>16}")
print("-" * 48)
for t in [0, 4, 9, 14, 19, 24, 29]:
    print(f"{t+1:>14} {trace_rnn[t]:>16.3e} {trace_lstm[t]:>16.3e}")

print(f"\nAfter 30 timesteps:")
print(f"  RNN  gradient: {trace_rnn[-1]:.3e}  -> vanished, early inputs forgotten")
print(f"  LSTM gradient: {trace_lstm[-1]:.3e}  -> preserved, long-range memory works")
print(f"\nThis is why LSTMs handle long sequences (long sentences, long audio)")
print(f"while vanilla RNNs struggle to learn long-range dependencies.")

=== Why Vanilla RNNs Forget: Gradient Flow Over Time ===

During BPTT (Backprop Through Time), the gradient at the first timestep
depends on a product of factors across ALL timesteps.

Vanilla RNN (recurrent weight W=0.8, sigmoid neurons):
  Gradient reaching t=1 after 30 steps: 1.074e-21
  (each step multiplies by W*sigmoid' = 0.2 < 1, so it vanishes)

LSTM (cell-state highway, forget gate f_t ~ 0.95):
  Gradient reaching t=1 after 30 steps: 2.146e-01
  (cell state path multiplies by ~f_t=0.95, close to 1, so it survives)

 Timestep back     RNN gradient    LSTM gradient
------------------------------------------------
             1        2.000e-01        9.500e-01
             5        3.200e-04        7.738e-01
            10        1.024e-07        5.987e-01
            15        3.277e-11        4.633e-01
            20        1.049e-14        3.585e-01
            25        3.355e-18        2.774e-01
            30        1.074e-21        2.146e-01

After 30 timesteps:
  RNN  g